# SCADA Lead-Time-Aware Anomaly Detection
## Master Experiment Notebook

**Run order:**
1. Cell 0 — Environment check
2. Cell 1 — Preprocessing (load raw IMS, save parquet)
3. Cell 2 — Feature extraction
4. Cell 3 — Baseline experiment (all methods)
5. Cell 4 — Threshold sweep (VLT vs FAR curve)
6. Cell 5 — Dashboard launch
7. Cell 6 — Uncertainty / conformal
8. Cell 7 — SHAP explainability


In [2]:
# ─── Cell 0: Environment setup ───────────────────────────────────────────────
import sys, os

# Colab compatibility
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/scada-leadtime-benchmark'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install plotly shap pyarrow ruptures -q
else:
    # Local VS Code — project root is one level up from notebooks/
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')   # headless — saves figures to file

from src.config import PATHS, DATASET, FEATURES, SPLIT, THRESHOLD, EXPERIMENT
print('Config loaded OK')
print(f'Raw IMS dir: {PATHS["raw_ims"]}')
print(f'Runs expected: {DATASET["runs"]}')

Project root: c:\scada


ImportError: DLL load failed while importing _partitioner: An Application Control policy has blocked this file.

In [2]:
# ─── Cell 1: Preprocessing ───────────────────────────────────────────────────
from src.preprocessing import run_preprocessing_pipeline

runs = run_preprocessing_pipeline(
    raw_ims_dir   = PATHS['raw_ims'],
    run_names     = DATASET['runs'],
    processed_dir = PATHS['processed'],
    failure_times = DATASET['failure_times'],
    n_channels    = DATASET['n_channels'],
    resample_freq = '1min',
)

for run_name, df in runs.items():
    print(f'{run_name}: {df.shape} | {df.index[0]} → {df.index[-1]}')

2026-06-04 09:14:50,068 [INFO] 
2026-06-04 09:14:50,068 [INFO] Processing run: 1st_test
2026-06-04 09:14:50,155 [INFO] Loading 2156 snapshot files from 1st_test ...
2026-06-04 09:15:04,566 [INFO]   Processed 500/2156 files ...
2026-06-04 09:15:18,587 [INFO]   Processed 1000/2156 files ...
2026-06-04 09:15:32,718 [INFO]   Processed 1500/2156 files ...
2026-06-04 09:15:45,078 [INFO]   Processed 2000/2156 files ...
2026-06-04 09:15:48,975 [INFO] Loaded 2156 snapshots, 48 features per snapshot.
2026-06-04 09:15:49,077 [INFO] Saved processed data → c:\scada\data\processed\1st_test_features.parquet
2026-06-04 09:15:49,078 [INFO] 
2026-06-04 09:15:49,078 [INFO] Processing run: 2nd_test
2026-06-04 09:15:49,104 [INFO] Loading 984 snapshot files from 2nd_test ...
2026-06-04 09:15:55,884 [INFO]   Processed 500/984 files ...
2026-06-04 09:16:02,008 [INFO] Loaded 984 snapshots, 24 features per snapshot.
2026-06-04 09:16:02,037 [INFO] Saved processed data → c:\scada\data\processed\2nd_test_features.

1st_test: (2155, 49) | 2003-10-22 12:06:00 → 2003-11-25 23:39:00
2nd_test: (984, 25) | 2004-02-12 10:32:00 → 2004-02-19 06:22:00
3rd_test: (6324, 25) | 2004-03-04 09:27:00 → 2004-04-18 02:42:00


In [7]:
# ─── Cell 2: Quick sanity-check plot ─────────────────────────────────────────
# Should see clear RMS rise in the last ~10-20% of the run
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RUN = '2nd_test'   # change to '1st_test' or '3rd_test' to inspect others
df = runs[RUN]
t_fail = pd.Timestamp(DATASET['failure_times'][RUN])

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# RMS of primary bearing channel
rms_col = 'rms_ch0'
axes[0].plot(df.index, df[rms_col], color='#42A5F5', linewidth=1.2)
axes[0].axvline(t_fail, color='red', linestyle='--', linewidth=2, label='Failure')
axes[0].set_title(f'{RUN} — RMS Ch0 over time', fontsize=12)
axes[0].set_ylabel('RMS')
axes[0].legend()

# Kurtosis
kurt_col = 'kurt_ch0'
axes[1].plot(df.index, df[kurt_col], color='#EF5350', linewidth=1.2)
axes[1].axvline(t_fail, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Kurtosis Ch0', fontsize=12)
axes[1].set_ylabel('Kurtosis')
axes[1].set_xlabel('Time')

plt.tight_layout()
out = os.path.join(PATHS['results_figures'], f'sanity_rms_{RUN}.png')
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved → {out}')
plt.close()

Saved → c:\scada\results\figures\sanity_rms_2nd_test.png


In [8]:
# ─── Cell 3: Full experiment — all methods ────────────────────────────────────
from src import load_pipeline, run_experiment

RUN = '2nd_test'
summary_df, all_results, pipe = run_experiment(RUN)

print('\n=== RESULTS TABLE ===')
print(summary_df.to_string(index=False))

# Save table
out_csv = os.path.join(PATHS['results_tables'], f'method_comparison_{RUN}.csv')
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f'\nSaved → {out_csv}')

2026-06-03 19:32:39,506 [INFO] Loaded processed data ← c:\scada\data\processed\2nd_test_features.parquet ((984, 25))
2026-06-03 19:32:41,827 [INFO] Feature extraction: 195 windows × 446 features (window_size=10, overlap=0.5)
2026-06-03 19:32:41,830 [INFO] Split: train=97, cal=20, test=78 (total=195)
2026-06-03 19:32:41,956 [INFO] Threshold [percentile]: 4.955721
2026-06-03 19:32:41,959 [INFO] [3σ Rule (σ=3.0)] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-03 19:32:41,961 [INFO] Threshold [percentile]: 1.096326
2026-06-03 19:32:41,962 [INFO] [EWMA (λ=0.2, k=3.0)] FAT=2004-02-16 22:52:00 | VLT=55.50h | FAR=0.0% | valid=True
2026-06-03 19:32:42,685 [INFO] Threshold [percentile]: 95.010209
2026-06-03 19:32:42,686 [INFO] [Hotelling T²] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-03 19:32:42,944 [INFO] IsolationForest fitted on X=(97, 445)
2026-06-03 19:32:42,993 [INFO] Threshold [percentile]: -0.021904
2026-06-03 19:32:42,994 [INFO] [Isola


=== RESULTS TABLE ===
             Method  VLT (hours)  FAR (%)  Valid Alarm                 FAT  Threshold
   Isolation Forest         58.0      0.0         True 2004-02-16 20:22:00    -0.0219
EWMA (λ=0.2, k=3.0)         55.5      0.0         True 2004-02-16 22:52:00     1.0963
    3σ Rule (σ=3.0)          0.0    100.0        False 2004-02-16 12:52:00     4.9557
       Hotelling T²          0.0    100.0        False 2004-02-16 12:52:00    95.0102

Saved → c:\scada\results\tables\method_comparison_2nd_test.csv


In [4]:
# ─── Cell 4: Plot alarm timelines for each method ─────────────────────────────
from src.lead_time import plot_alarm_timeline, plot_lead_time_comparison

for result in all_results:
    fig = plot_alarm_timeline(
        result,
        save_path=os.path.join(
            PATHS['results_figures'],
            f'alarm_timeline_{RUN}_{result["short_name"]}.png'
        )
    )
    plt.close()

# Comparison chart
fig = plot_lead_time_comparison(
    summary_df,
    save_path=os.path.join(PATHS['results_figures'], f'vlt_comparison_{RUN}.png')
)
plt.close()
print('Alarm timeline plots saved.')

2026-06-03 18:11:21,067 [INFO] Saved alarm timeline → c:\scada\results\figures\alarm_timeline_2nd_test_three_sigma.png
2026-06-03 18:11:21,227 [INFO] Saved alarm timeline → c:\scada\results\figures\alarm_timeline_2nd_test_ewma.png
2026-06-03 18:11:21,365 [INFO] Saved alarm timeline → c:\scada\results\figures\alarm_timeline_2nd_test_hotelling_t2.png
2026-06-03 18:11:21,535 [INFO] Saved alarm timeline → c:\scada\results\figures\alarm_timeline_2nd_test_isolation_forest.png
2026-06-03 18:11:21,703 [INFO] Saved comparison chart → c:\scada\results\figures\vlt_comparison_2nd_test.png


Alarm timeline plots saved.


In [5]:
# ─── Cell 5: Threshold sweep (VLT vs FAR curve) ───────────────────────────────
from src.thresholds import threshold_sweep
from src.lead_time import plot_vlt_vs_far
from src.models import IsolationForestDetector
from src.config import THRESHOLD

pipe = load_pipeline(RUN)
det = IsolationForestDetector(n_estimators=200, random_state=42)
det.fit(pipe['X_train'])

scores_train = det.score(pipe['X_train'])
scores_test  = det.score(pipe['X_test'])
ts_test      = pipe['ts_test']
t_fail_str   = DATASET['failure_times'][RUN]
t_normal_end_str = str(pipe['t_normal_end'])

sweep_df = threshold_sweep(
    scores_train     = scores_train,
    scores_test      = scores_test,
    timestamps_test  = ts_test,
    failure_time     = t_fail_str,
    normal_period_end= t_normal_end_str,
    percentile_range = THRESHOLD['sweep_percentiles'],
    persistence      = THRESHOLD['alarm_persistence'],
)

print(sweep_df.to_string(index=False))

fig = plot_vlt_vs_far(
    sweep_df,
    method_name='Isolation Forest',
    save_path=os.path.join(PATHS['results_figures'], f'vlt_vs_far_{RUN}.png')
)
plt.close()

sweep_df.to_csv(
    os.path.join(PATHS['results_tables'], f'threshold_sweep_{RUN}.csv'),
    index=False
)
print('Threshold sweep done.')

2026-06-03 18:12:14,585 [INFO] Loaded processed data ← c:\scada\data\processed\2nd_test_features.parquet ((984, 25))
2026-06-03 18:12:16,211 [INFO] Feature extraction: 195 windows × 446 features (window_size=10, overlap=0.5)
2026-06-03 18:12:16,214 [INFO] Split: train=97, cal=20, test=78 (total=195)
2026-06-03 18:12:16,628 [INFO] IsolationForest fitted on X=(97, 445)
2026-06-03 18:12:16,649 [INFO] Threshold [percentile]: -0.030802
2026-06-03 18:12:16,650 [INFO]   p= 90.0% | thresh=-0.0308 | VLT=58.83h | FAR=0.0%
2026-06-03 18:12:16,651 [INFO] Threshold [percentile]: -0.029779
2026-06-03 18:12:16,652 [INFO]   p= 92.5% | thresh=-0.0298 | VLT=58.83h | FAR=0.0%
2026-06-03 18:12:16,653 [INFO] Threshold [percentile]: -0.025853
2026-06-03 18:12:16,655 [INFO]   p= 95.0% | thresh=-0.0259 | VLT=58.00h | FAR=0.0%
2026-06-03 18:12:16,657 [INFO] Threshold [percentile]: -0.021904
2026-06-03 18:12:16,658 [INFO]   p= 97.5% | thresh=-0.0219 | VLT=58.00h | FAR=0.0%
2026-06-03 18:12:16,659 [INFO] Thresho

 percentile  threshold                 FAT  VLT_hours  FAR_pct  valid_alarm
       90.0  -0.030802 2004-02-16 19:32:00  58.833333      0.0         True
       92.5  -0.029779 2004-02-16 19:32:00  58.833333      0.0         True
       95.0  -0.025853 2004-02-16 20:22:00  58.000000      0.0         True
       97.5  -0.021904 2004-02-16 20:22:00  58.000000      0.0         True
       99.0  -0.014480 2004-02-16 20:22:00  58.000000      0.0         True
       99.5   0.052214 2004-02-18 15:42:00  14.666667      0.0         True
Threshold sweep done.


In [4]:
from src import load_pipeline, run_experiment
from src.config import DATASET

summary_df, all_results, pipe = run_experiment("2nd_test")
result = next((r for r in all_results if r["short_name"] == "isolation_forest"), all_results[0])

df = pipe["df_full"].copy()

# Check what columns exist
print("Columns:", list(df.columns)[:6])
print("Shape:", df.shape)

# Check if RMS columns exist
rms_cols = [c for c in df.columns if c.startswith("rms_")]
print("RMS cols:", rms_cols)

# Check scores
print("Scores shape:", result["scores_test"].shape)
print("Timestamps:", result["timestamps"][:3])

2026-06-04 09:11:01,789 [INFO] Loaded processed data ← c:\scada\data\processed\2nd_test_features.parquet ((984, 25))
2026-06-04 09:11:03,393 [INFO] Feature extraction: 195 windows × 446 features (window_size=10, overlap=0.5)
2026-06-04 09:11:03,396 [INFO] Split: train=97, cal=20, test=78 (total=195)
2026-06-04 09:11:03,486 [INFO] Threshold [percentile]: 4.955721
2026-06-04 09:11:03,487 [INFO] [3σ Rule (σ=3.0)] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-04 09:11:03,488 [INFO] Threshold [percentile]: 1.096326
2026-06-04 09:11:03,489 [INFO] [EWMA (λ=0.2, k=3.0)] FAT=2004-02-16 22:52:00 | VLT=55.50h | FAR=0.0% | valid=True
2026-06-04 09:11:03,562 [INFO] Threshold [percentile]: 95.010209
2026-06-04 09:11:03,564 [INFO] [Hotelling T²] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-04 09:11:03,764 [INFO] IsolationForest fitted on X=(97, 445)
2026-06-04 09:11:03,787 [INFO] Threshold [percentile]: -0.021904
2026-06-04 09:11:03,788 [INFO] [Isola

Columns: ['rms_ch0', 'kurt_ch0', 'skew_ch0', 'p2p_ch0', 'crest_ch0', 'var_ch0']
Shape: (984, 25)
RMS cols: ['rms_ch0', 'rms_ch1', 'rms_ch2', 'rms_ch3']
Scores shape: (78,)
Timestamps: DatetimeIndex(['2004-02-16 12:52:00', '2004-02-16 13:42:00',
               '2004-02-16 14:32:00'],
              dtype='datetime64[us]', freq=None)


In [3]:
#cell 6
import os
os.chdir(r"C:\scada")

# Clear pycache first
import importlib
import src.dashboard
importlib.reload(src.dashboard)

from src.dashboard import launch_dashboard
launch_dashboard("2nd_test", "isolation_forest", port=8050, debug=True)

2026-06-04 09:17:54,148 [INFO] Loaded processed data ← c:\scada\data\processed\2nd_test_features.parquet ((984, 25))



───────────────────────────────────────────────────────
  Loading experiment: 2nd_test
───────────────────────────────────────────────────────


2026-06-04 09:17:56,118 [INFO] Feature extraction: 195 windows × 446 features (window_size=10, overlap=0.5)
2026-06-04 09:17:56,122 [INFO] Split: train=97, cal=20, test=78 (total=195)
2026-06-04 09:17:56,236 [INFO] Threshold [percentile]: 4.955721
2026-06-04 09:17:56,238 [INFO] [3σ Rule (σ=3.0)] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-04 09:17:56,239 [INFO] Threshold [percentile]: 1.096326
2026-06-04 09:17:56,240 [INFO] [EWMA (λ=0.2, k=3.0)] FAT=2004-02-16 22:52:00 | VLT=55.50h | FAR=0.0% | valid=True
2026-06-04 09:17:56,364 [INFO] Threshold [percentile]: 95.010209
2026-06-04 09:17:56,365 [INFO] [Hotelling T²] FAT=2004-02-16 12:52:00 | VLT=0.00h | FAR=100.0% | valid=False
2026-06-04 09:17:56,646 [INFO] IsolationForest fitted on X=(97, 445)
2026-06-04 09:17:56,674 [INFO] Threshold [percentile]: -0.021904
2026-06-04 09:17:56,675 [INFO] [Isolation Forest] FAT=2004-02-16 20:22:00 | VLT=58.00h | FAR=0.0% | valid=True



  Dashboard →  http://127.0.0.1:8050
  Ctrl+C to stop



In [4]:
# ─── Cell 7: Conformal detector ──────────────────────────────────────────────
from src.uncertainty import ConformalDetector
from src.models import IsolationForestDetector
from src.lead_time import compute_FAT, compute_VLT, compute_FAR

pipe = load_pipeline(RUN)

base_det = IsolationForestDetector(n_estimators=200, random_state=42)
conformal = ConformalDetector(base_detector=base_det, alpha=0.05)

conformal.calibrate(pipe['X_train'], pipe['X_cal'])

alarm_conf, p_vals, scores_conf = conformal.alarm(
    pipe['X_test'], persistence=THRESHOLD['alarm_persistence']
)

ts_test  = pipe['ts_test']
t_fail   = pd.Timestamp(DATASET['failure_times'][RUN])

fat_conf = compute_FAT(alarm_conf, ts_test)
vlt_conf = compute_VLT(fat_conf, t_fail, pipe['t_normal_end'])
far_conf = compute_FAR(alarm_conf, ts_test, pipe['t_normal_end'])

print(f'Conformal IF  — FAT: {fat_conf} | VLT: {vlt_conf:.2f}h | FAR: {far_conf*100:.1f}%')

NameError: name 'load_pipeline' is not defined

In [ ]:
# ─── Cell 8: SHAP explanation at alarm time ───────────────────────────────────
from src.uncertainty import compute_shap_values, plot_shap_waterfall_at_alarm
from src.models import IsolationForestDetector

pipe = load_pipeline(RUN)

det_shap = IsolationForestDetector(n_estimators=200, random_state=42)
det_shap.fit(pipe['X_train'])

scores_test = det_shap.score(pipe['X_test'])

# Get window index near alarm time (use top scoring window)
alarm_window_idx = int(np.argmax(scores_test))

shap_vals = compute_shap_values(
    det_shap, pipe['X_test'], pipe['feature_names'], max_samples=150
)

fig = plot_shap_waterfall_at_alarm(
    shap_values      = shap_vals,
    X                = pipe['X_test'],
    feature_names    = pipe['feature_names'],
    alarm_window_idx = alarm_window_idx,
    save_path        = os.path.join(PATHS['results_figures'], f'shap_alarm_{RUN}.png'),
)
plt.close()
print('SHAP explanation saved.')

In [1]:
# Diagnostic — see exactly what we are working with
from src import load_pipeline, run_experiment
from src.config import DATASET, THRESHOLD, PATHS
import pandas as pd
import os

# Check 1: Which runs have processed parquet files
print("=== PROCESSED FILES ===")
processed_dir = PATHS["processed"]
for f in os.listdir(processed_dir):
    path = os.path.join(processed_dir, f)
    size = os.path.getsize(path) / 1024
    print(f"  {f}  ({size:.1f} KB)")

# Check 2: Current config values
print("\n=== CURRENT CONFIG ===")
print(f"  threshold_strategy:  {THRESHOLD['strategy']}")
print(f"  threshold_percentile: {THRESHOLD['percentile']}")
print(f"  alarm_persistence:   {THRESHOLD['alarm_persistence']}")
print(f"  sweep_percentiles:   {THRESHOLD['sweep_percentiles']}")

# Check 3: Failure times
print("\n=== FAILURE TIMES ===")
for run, t in DATASET["failure_times"].items():
    print(f"  {run}: {t}")

ModuleNotFoundError: No module named 'src'